# Pandas 数据分析实战

本notebook系统讲解Pandas数据分析的核心技术，包括数据导入导出、缺失值处理、时间数据操作、数据类型转换、重复值处理、数据分箱、字符串处理以及特征工程等重要知识点。通过丰富的代码示例和详细的中文注释，帮助读者掌握数据分析的基本技能，为后续的深入学习打下坚实基础。

---

## 1. 数据导入与导出

在数据分析工作中，我们首先需要将外部数据加载到Python环境中。Pandas支持多种数据格式的读取，最常用的是CSV文件和JSON文件。通过`read_csv`函数可以轻松读取CSV文件，而`read_json`则用于读取JSON格式的数据。读取后的数据会被转换为DataFrame结构，这是一种二维表格数据结构，非常适合进行数据分析工作。

数据的导出同样重要，当我们完成数据处理后，通常需要将结果保存为文件格式供后续使用。`to_csv`方法可以将DataFrame保存为CSV文件，`to_json`则可以保存为JSON格式。在导出时，我们可以指定各种参数来控制文件的格式，如是否包含索引、分隔符设置等。

In [1]:
# 数据导入
import pandas as pd

# 读取CSV文件
df = pd.read_csv('data/employees.csv')
print("数据类型:", type(df))
print("\n最后5行数据:")
print(df.tail())
print("\n平均薪资:", df['salary'].mean())

数据类型: <class 'pandas.DataFrame'>

最后5行数据:
     employee_id first_name last_name     email  phone_number      job_id  \
102          202        Pat       Fay      PFAY  603.123.6666      MK_REP   
103          203      Susan    Mavris   SMAVRIS  515.123.7777      HR_REP   
104          204    Hermann      Baer     HBAER  515.123.8888      PR_REP   
105          205    Shelley   Higgins  SHIGGINS  515.123.8080      AC_MGR   
106          206    William     Gietz    WGIETZ  515.123.8181  AC_ACCOUNT   

      salary  commission_pct  manager_id  department_id  
102   6000.0             NaN       201.0           20.0  
103   6500.0             NaN       101.0           40.0  
104  10000.0             NaN       101.0           70.0  
105  12000.0             NaN       101.0          110.0  
106   8300.0             NaN       205.0          110.0  

平均薪资: 6461.682242990654


In [2]:
# 数据导出
df_export = df.tail()
df_export.to_csv('data/new.csv')  # 保存为CSV文件
print("数据已导出到 data/new.csv")

数据已导出到 data/new.csv


In [3]:
# JSON文件读取
df_json = pd.read_json('data/data1.json')
print("JSON数据类型:", type(df_json))

# 通过json模块读取复杂JSON结构
import json
with open('data/test.json',encoding='utf-8') as f:
    data = json.load(f)

print("JSON数据总类型:", type(data))

# 从嵌套JSON中提取特定数据
df_from_json = pd.DataFrame(data['users'])
print("\n从users字段提取的DataFrame:")
df_from_json

JSON数据类型: <class 'pandas.DataFrame'>
JSON数据总类型: <class 'dict'>

从users字段提取的DataFrame:


,id,name,age,email,is_active,join_date
0,1,张三,28,zhangsan@example.com,True,2022-03-15
1,2,李四,35,lisi@example.com,False,2021-11-02
2,3,王五,24,wangwu@example.com,True,2023-01-20


---

## 2. 缺失值处理

缺失值是数据分析中非常常见的问题，可能由于数据采集不完整、存储错误或数据传输过程中丢失等原因造成。Pandas使用NaN（Not a Number）来表示缺失值，对于不同的数据类型，缺失值的表示方式也可能不同，如None、pd.NA等。

处理缺失值的第一步是识别缺失值的位置。`isna()`和`isnull()`方法可以检测缺失值，返回一个布尔值的DataFrame。我们还可以使用`isna().sum()`来统计每列缺失值的数量。

对于缺失值的处理策略，主要有两种：一是删除缺失值所在的行或列，二是用特定值填充缺失值。删除方法包括`dropna()`，可以通过`how`参数设置删除条件（如`how='all'`表示只删除全为缺失值的行），通过`thresh`参数设置非缺失值的最小数量，通过`subset`参数指定要检查的列。填充方法包括`fillna()`，可以用常数填充、用均值填充、前向填充（ffill，使用前一个值填充）和后向填充（bfill，使用后一个值填充）。

In [4]:
# 缺失值的识别与检测
import pandas as pd
import numpy as np

# 创建包含缺失值的数据
s = pd.Series([12, 25, np.nan, None, pd.NA])
df = pd.DataFrame(
    [[1, pd.NA, 2], [2, 3, 5], [None, 4, 6]],
    columns=['第1列', '第2列', '第3列']
)

print("原始序列:")
print(s)

# 检测缺失值
print("\nisna()检测结果:")
print(s.isna())

print("\nDataFrame缺失值检测:")
print(df.isna())

# 统计缺失值数量
print("\n每行缺失值数量:")
print(df.isna().sum(axis=1))
print("\n序列缺失值总数:", s.isna().sum())

原始序列:
0      12
1      25
2     NaN
3    None
4    <NA>
dtype: object

isna()检测结果:
0    False
1    False
2     True
3     True
4     True
dtype: bool

DataFrame缺失值检测:
     第1列    第2列    第3列
0  False   True  False
1  False  False  False
2   True  False  False

每行缺失值数量:
0    1
1    0
2    1
dtype: int64

序列缺失值总数: 3


In [5]:
# 删除缺失值
print("原始DataFrame:")
print(df)

print("\ndropna()删除所有含缺失值的行:")
print(df.dropna())

print("\ndropna(how='all')只删除全为缺失值的行:")
print(df.dropna(how='all'))

print("\ndropna(thresh=1)保留至少有1个非缺失值的行:")
print(df.dropna(thresh=1))

print("\ndropna(axis=1)删除含缺失值的列:")
print(df.dropna(axis=1))

print("\ndropna(subset=['第1列'])删除第1列有缺失值的行:")
print(df.dropna(subset=['第1列']))

原始DataFrame:
   第1列   第2列  第3列
0  1.0  <NA>    2
1  2.0     3    5
2  NaN     4    6

dropna()删除所有含缺失值的行:
   第1列 第2列  第3列
1  2.0   3    5

dropna(how='all')只删除全为缺失值的行:
   第1列   第2列  第3列
0  1.0  <NA>    2
1  2.0     3    5
2  NaN     4    6

dropna(thresh=1)保留至少有1个非缺失值的行:
   第1列   第2列  第3列
0  1.0  <NA>    2
1  2.0     3    5
2  NaN     4    6

dropna(axis=1)删除含缺失值的列:
   第3列
0    2
1    5
2    6

dropna(subset=['第1列'])删除第1列有缺失值的行:
   第1列   第2列  第3列
0  1.0  <NA>    2
1  2.0     3    5


In [6]:
# 填充缺失值
df_weather = pd.read_csv('data/weather_withna.csv')
print("天气数据缺失值统计:")
print(df_weather.isna().sum(axis=0))
print("\n天气数据前5行:")
print(df_weather.head())

# 使用字典填充不同的列
print("\n使用指定值填充:")
print(df_weather.fillna({'temp_max': 20, 'wind': 2.5}).tail())

# 使用均值填充
print("\n使用均值填充:")
print(df_weather.fillna(df_weather[['temp_max', 'wind']].mean()).tail())

# 前向填充（用前一个值填充）
print("\n前向填充(ffill):")
print(df_weather.ffill().tail())

# 后向填充（用后一个值填充）
print("\n后向填充(bfill):")
print(df_weather.bfill().tail())

天气数据缺失值统计:
date               0
precipitation    303
temp_max         303
temp_min         303
wind             303
weather          303
dtype: int64

天气数据前5行:
         date  precipitation  temp_max  temp_min  wind  weather
0  2012-01-01            0.0      12.8       5.0   4.7  drizzle
1  2012-01-02           10.9      10.6       2.8   4.5     rain
2  2012-01-03            0.8      11.7       7.2   2.3     rain
3  2012-01-04           20.3      12.2       5.6   4.7     rain
4  2012-01-05            1.3       8.9       2.8   6.1     rain

使用指定值填充:
            date  precipitation  temp_max  temp_min  wind weather
1456  2015-12-27            NaN      20.0       NaN   2.5     NaN
1457  2015-12-28            NaN      20.0       NaN   2.5     NaN
1458  2015-12-29            NaN      20.0       NaN   2.5     NaN
1459  2015-12-30            NaN      20.0       NaN   2.5     NaN
1460  2015-12-31           20.6      12.2       5.0   3.8    rain

使用均值填充:
            date  precipitation   temp_ma

---

## 3. 时间数据处理

时间数据是数据分析中的重要类型，Pandas提供了强大的时间处理功能。`Timestamp`是Pandas中表示单个时间点的对象，它包含了年、月、日、时、分、秒等详细信息。通过Timestamp对象的属性，我们可以轻松提取时间的各个组成部分，如年份、月份、季度等。

`to_datetime`函数用于将字符串或其他格式的数据转换为Timestamp类型，这是处理时间数据的第一步。转换后的Timestamp列可以使用`.dt`访问器进行各种时间相关的操作，如提取年月日、计算时间差、获取星期几等。

`date_range`函数用于生成日期序列，可以指定起始日期、结束日期和频率。频率可以是日（D）、周（W）、月（M）、季度（Q）、年（Y）等。`resample`方法则用于对时间序列数据进行重采样，可以将高频数据聚合为低频数据，或将低频数据插值为高频数据，这在处理时间序列数据时非常有用。

In [7]:
# Timestamp时间戳对象
import pandas as pd

d = pd.Timestamp('2015-02-28 10:22')
d1 = pd.Timestamp('2015-02-28 13:22')

print("时间戳:", d)
print("类型:", type(d))

# 提取时间组成部分
print("\n年:", d.year)
print("月:", d.month)
print("日:", d.day)
print("时:", d.hour, "分:", d.minute, "秒:", d.second)
print("季度:", d.quarter)
print("是否是月底:", d.is_month_end)

# 时间戳方法
print("\n星期几:", d.day_name())
print("转换为天数:", d.to_period("D"))
print("转换为季度:", d1.to_period("Q"))
print("转换为年度:", d1.to_period("Y"))
print("转换为月度:", d1.to_period("M"))
print("转换为周:", d1.to_period("W"))

时间戳: 2015-02-28 10:22:00
类型: <class 'pandas.Timestamp'>

年: 2015
月: 2
日: 28
时: 10 分: 22 秒: 0
季度: 1
是否是月底: True

星期几: Saturday
转换为天数: 2015-02-28
转换为季度: 2015Q1
转换为年度: 2015
转换为月度: 2015-02
转换为周: 2015-02-23/2015-03-01


In [8]:
# 字符串转换为日期类型
a = pd.to_datetime('20150228')
print("字符串转日期:", a)
print("类型:", type(a))
print("星期:", a.day_name())

# DataFrame中的日期转换
df = pd.DataFrame({
    'sales': [100, 200, 300],
    'date': ['20250601', '20250602', '20250603']
})
df['datetime'] = pd.to_datetime(df['date'])

print("\n转换后DataFrame:")
print(df.info())

# 从datetime提取时间属性
df['week'] = df['datetime'].dt.day_name()
df['year'] = df['datetime'].dt.year
print("\n添加周和年列:")
df

字符串转日期: 2015-02-28 00:00:00
类型: <class 'pandas.Timestamp'>
星期: Saturday

转换后DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   sales     3 non-null      int64         
 1   date      3 non-null      str           
 2   datetime  3 non-null      datetime64[us]
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 204.0 bytes
None

添加周和年列:


,sales,date,datetime,week,year
0,100,20250601,2025-06-01,Sunday,2025
1,200,20250602,2025-06-02,Monday,2025
2,300,20250603,2025-06-03,Tuesday,2025


In [9]:
# CSV文件中直接解析日期
df = pd.read_csv('data/weather.csv', parse_dates=['date'])
print("CSV解析后的数据类型:")
print(df.info())

print("\n提取星期几:")
print(df['date'].dt.day_name())

CSV解析后的数据类型:
<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           1461 non-null   datetime64[us]
 1   precipitation  1461 non-null   float64       
 2   temp_max       1461 non-null   float64       
 3   temp_min       1461 non-null   float64       
 4   wind           1461 non-null   float64       
 5   weather        1461 non-null   str           
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 68.6 KB
None

提取星期几:
0          Sunday
1          Monday
2         Tuesday
3       Wednesday
4        Thursday
          ...    
1456       Sunday
1457       Monday
1458      Tuesday
1459    Wednesday
1460     Thursday
Name: date, Length: 1461, dtype: str


In [10]:
# 时间间隔计算
d1 = pd.Timestamp('2013-01-15')
d2 = pd.Timestamp('2023-02-23')
delta = d2 - d1

print("时间间隔类型:", type(delta))
print("间隔天数:", delta)

# 使用时间间隔作为索引
df = pd.read_csv('data/weather.csv', parse_dates=['date'])
df['delta'] = df['date'] - df['date'][0]
df.set_index('delta', inplace=True)
print("\n使用时间间隔作为索引:")
print(df.loc['10 days':'20 days'])

时间间隔类型: <class 'pandas.Timedelta'>
间隔天数: 3691 days 00:00:00

使用时间间隔作为索引:
              date  precipitation  temp_max  temp_min  wind weather
delta                                                              
10 days 2012-01-11            0.0       6.1      -1.1   5.1     sun
11 days 2012-01-12            0.0       6.1      -1.7   1.9     sun
12 days 2012-01-13            0.0       5.0      -2.8   1.3     sun
13 days 2012-01-14            4.1       4.4       0.6   5.3    snow
14 days 2012-01-15            5.3       1.1      -3.3   3.2    snow
15 days 2012-01-16            2.5       1.7      -2.8   5.0    snow
16 days 2012-01-17            8.1       3.3       0.0   5.6    snow
17 days 2012-01-18           19.8       0.0      -2.8   5.0    snow
18 days 2012-01-19           15.2      -1.1      -2.8   1.6    snow
19 days 2012-01-20           13.5       7.2      -1.1   2.3    snow
20 days 2012-01-21            3.0       8.3       3.3   8.2    rain


In [11]:
# date_range生成日期序列
# 按周生成
days_weekly = pd.date_range("2025-07-03", "2026-02-09", freq="W")
print("按周生成的日期:")
print(days_weekly)

# 按季度生成，指定周期数
days_quarterly = pd.date_range("2025-07-03", periods=10, freq="QE")
print("\n按季度生成的日期:")
print(days_quarterly)

按周生成的日期:
DatetimeIndex(['2025-07-06', '2025-07-13', '2025-07-20', '2025-07-27',
               '2025-08-03', '2025-08-10', '2025-08-17', '2025-08-24',
               '2025-08-31', '2025-09-07', '2025-09-14', '2025-09-21',
               '2025-09-28', '2025-10-05', '2025-10-12', '2025-10-19',
               '2025-10-26', '2025-11-02', '2025-11-09', '2025-11-16',
               '2025-11-23', '2025-11-30', '2025-12-07', '2025-12-14',
               '2025-12-21', '2025-12-28', '2026-01-04', '2026-01-11',
               '2026-01-18', '2026-01-25', '2026-02-01', '2026-02-08'],
              dtype='datetime64[us]', freq='W-SUN')

按季度生成的日期:
DatetimeIndex(['2025-09-30', '2025-12-31', '2026-03-31', '2026-06-30',
               '2026-09-30', '2026-12-31', '2027-03-31', '2027-06-30',
               '2027-09-30', '2027-12-31'],
              dtype='datetime64[us]', freq='QE-DEC')


In [12]:
# resample重采样
df = pd.read_csv('data/weather.csv', parse_dates=['date'])
df.set_index('date', inplace=True)

# 按月采样计算均值
print("按月采样(MS=Month Start):")
print(df[['temp_max', 'temp_min']].resample('MS').mean())

# 按年采样计算均值
print("\n按年采样(YE=Year End):")
print(df[['temp_max', 'temp_min']].resample('YE').mean())

按月采样(MS=Month Start):
             temp_max   temp_min
date                            
2012-01-01   7.054839   1.541935
2012-02-01   9.275862   3.203448
2012-03-01   9.554839   2.838710
2012-04-01  14.873333   5.993333
2012-05-01  17.661290   8.190323
2012-06-01  18.693333  10.480000
2012-07-01  22.906452  12.932258
2012-08-01  25.858065  14.009677
2012-09-01  22.880000  11.243333
2012-10-01  15.829032   8.380645
2012-11-01  11.326667   5.226667
2012-12-01   7.235484   3.293548
2013-01-01   6.106452   0.796774
2013-02-01   9.467857   4.325000
2013-03-01  12.709677   4.977419
2013-04-01  14.243333   6.696667
2013-05-01  19.625806   9.922581
2013-06-01  23.253333  13.163333
2013-07-01  26.093548  13.932258
2013-08-01  26.119355  15.480645
2013-09-01  21.360000  13.590000
2013-10-01  14.229032   7.638710
2013-11-01  12.053333   5.590000
2013-12-01   7.022581   1.570968
2014-01-01   9.600000   4.096774
2014-02-01   8.200000   2.635714
2014-03-01  12.906452   5.425806
2014-04-01  15.460000

---

## 4. 数据类型转换

Pandas支持丰富的数据类型系统，包括整型、浮点型、字符串、日期时间、布尔型和类别型等。正确的数据类型对于数据存储效率和操作性能都很重要。`astype`方法可以用于数据类型之间的转换，例如将浮点数转换为整数，将字符串转换为日期等。

类别型（category）是一种特殊的数据类型，当数据中有大量重复的字符串值时，使用类别型可以显著减少内存占用并提高处理速度。将字符串列转换为类别型后，Pandas会在内部使用整数编码来表示这些值，这对于机器学习算法的训练特别有用。

`map`函数是另一个常用的转换工具，它可以根据一个映射字典来转换列的值，常用于将字符串编码转换为数值编码或将分类数据转换为二进制标志。

In [13]:
# 数据类型转换
df = pd.read_csv('data/sleep.csv')
print("原始数据类型:")
print(df.dtypes)

# 转换为整型
df['age'] = df['age'].astype('int16')
print("\nage列转换为int16后:")
print(df['age'].dtype)

# 转换为类别型
df['gender'] = df['gender'].astype('category')
print("\ngender列转换为category后:")
print(df['gender'].dtype)
print("类别型表示:", df['gender'])

原始数据类型:
person_id                    int64
gender                         str
age                          int64
occupation                     str
sleep_duration             float64
sleep_quality              float64
physical_activity_level      int64
stress_level                 int64
bmi_category                   str
blood_pressure                 str
heart_rate                   int64
daily_steps                  int64
sleep_disorder                 str
dtype: object

age列转换为int16后:
int16

gender列转换为category后:
category
类别型表示: 0        Male
1      Female
2        Male
3        Male
4        Male
        ...  
395    Female
396    Female
397    Female
398    Female
399      Male
Name: gender, Length: 400, dtype: category
Categories (2, str): ['Female', 'Male']


In [14]:
# map映射转换
df['is_male'] = df['gender'].map({'Female': True, 'Male': False})
print("使用map创建二值列:")
print(df['is_male'])

使用map创建二值列:
0      False
1       True
2      False
3      False
4      False
       ...  
395     True
396     True
397     True
398     True
399    False
Name: is_male, Length: 400, dtype: category
Categories (2, bool): [True, False]


---

## 5. 重复值处理

数据中可能存在重复的记录，这会影响分析结果的准确性。Pandas提供了`duplicated`方法用于检测重复行，返回一个布尔Series，其中True表示该行是重复的。默认情况下，只有当一整行所有列的值都完全相同时，才会被标记为重复。

`drop_duplicates`方法用于删除重复行。可以通过`subset`参数指定基于哪些列来判断重复，默认情况下会考虑所有列。`keep`参数控制保留哪一条重复记录，可以是'first'（保留第一条）、'last'（保留最后一条）或False（删除所有重复行）。在清洗数据时，通常会根据业务逻辑选择合适的策略处理重复值。

In [15]:
# 重复值处理
data = {
    "name": ['alice', 'alice', 'bob', 'alice', 'jack', 'bob'],
    "age": [26, 25, 30, 25, 35, 30],
    'city': ['NY', 'NY', 'LA', 'NY', 'SF', 'LA']
}
df = pd.DataFrame(data)

print("原始数据:")
print(df)

# 检测重复行
print("\n重复行检测:")
print(df.duplicated())

# 基于name列去重
print("\n基于name列去重:")
print(df.drop_duplicates(subset=['name']))

# 保留最后一次出现的行
print("\n保留最后一次出现的行:")
print(df.drop_duplicates(subset=['name'], keep='last'))

原始数据:
    name  age city
0  alice   26   NY
1  alice   25   NY
2    bob   30   LA
3  alice   25   NY
4   jack   35   SF
5    bob   30   LA

重复行检测:
0    False
1    False
2    False
3     True
4    False
5     True
dtype: bool

基于name列去重:
    name  age city
0  alice   26   NY
2    bob   30   LA
4   jack   35   SF

保留最后一次出现的行:
    name  age city
3  alice   25   NY
4   jack   35   SF
5    bob   30   LA


---

## 6. 数据分箱

数据分箱（Binning）是将连续变量离散化的一种技术，在数据分析中有着广泛的应用。例如，我们可以将年龄分成"青少年"、"中年"、"老年"等类别，将收入分成"低收入"、"中等收入"、"高收入"等区间。分箱后的数据更容易理解和解释，也便于进行分类型分析。

Pandas提供了两个重要的分箱函数：`cut`和`qcut`。`cut`函数按照数据的取值范围均匀切分，生成的每个区间宽度相等；`qcut`函数则按照数据的分位数切分，确保每个类别中的样本数量大致相等。对于分布不均匀的数据，`qcut`通常能提供更平衡的结果。

分箱函数返回的是一个Categorical对象，包含区间边界信息和类别标签。我们可以通过`labels`参数指定类别名称，通过`bins`参数指定分箱的边界，可以是整数（表示均分数量）或列表（指定具体边界）。

In [16]:
# 数据分箱
df = pd.read_csv('data/employees.csv')
df1 = df.head(10)[['employee_id', 'salary']]
print("员工薪资数据:")
print(df1)

# bins=3 将数据均分为3段
print("\n均分3段:")
print(pd.cut(df1['salary'], bins=3))
print("\n各段样本数量:")
print(pd.cut(df1['salary'], bins=3).value_counts())

# 指定分箱边界
print("\n指定边界[0,10000,20000,30000]:")
print(pd.cut(df1['salary'], bins=[0, 10000, 20000, 30000]).value_counts())

# 添加标签
df1['收入范围'] = pd.cut(
    df1['salary'],
    bins=[0, 10000, 20000, 30000],
    labels=['低', '中', '高']
)
print("\n添加收入范围标签后:")
print(df1)

员工薪资数据:
   employee_id   salary
0          100  24000.0
1          101  17000.0
2          102  17000.0
3          103   9000.0
4          104   6000.0
5          105   4800.0
6          106   4800.0
7          107   4200.0
8          108  12000.0
9          109   9000.0

均分3段:
0    (17400.0, 24000.0]
1    (10800.0, 17400.0]
2    (10800.0, 17400.0]
3     (4180.2, 10800.0]
4     (4180.2, 10800.0]
5     (4180.2, 10800.0]
6     (4180.2, 10800.0]
7     (4180.2, 10800.0]
8    (10800.0, 17400.0]
9     (4180.2, 10800.0]
Name: salary, dtype: category
Categories (3, interval[float64, right]): [(4180.2, 10800.0] < (10800.0, 17400.0] < (17400.0, 24000.0]]

各段样本数量:
salary
(4180.2, 10800.0]     6
(10800.0, 17400.0]    3
(17400.0, 24000.0]    1
Name: count, dtype: int64

指定边界[0,10000,20000,30000]:
salary
(0, 10000]        6
(10000, 20000]    3
(20000, 30000]    1
Name: count, dtype: int64

添加收入范围标签后:
   employee_id   salary 收入范围
0          100  24000.0    高
1          101  17000.0    中
2          10

In [17]:
# qcut按分位数分箱
print("按分位数分为3等份:")
print(pd.qcut(df1['salary'], 3).value_counts())

# 睡眠质量分箱
df = pd.read_csv('data/sleep.csv')
labels = ['差', '中', '优']
df['睡眠质量'] = pd.cut(df['sleep_quality'], bins=3, labels=labels)
print("\n睡眠质量分箱结果:")
print(df['睡眠质量'].value_counts())

# 查看分箱后的数据类型
print("\n分箱后数据类型:")
print("gender:", df['gender'].dtype)
print("睡眠质量:", df['睡眠质量'].dtype)

按分位数分为3等份:
salary
(4199.999, 6000.0]    4
(6000.0, 12000.0]     3
(12000.0, 24000.0]    3
Name: count, dtype: int64

睡眠质量分箱结果:
睡眠质量
中    206
优    129
差     65
Name: count, dtype: int64

分箱后数据类型:
gender: str
睡眠质量: category


---

## 7. 字符串处理

在实际数据分析中，很多原始数据是以字符串形式存储的，如姓名、地址、日期等。Pandas的字符串 accessor（通过`.str`访问）提供了丰富的字符串处理功能，包括分割、提取、替换、匹配等操作。

`str.split`方法可以将字符串按照分隔符分割成多个部分，返回一个列表列。通过`expand=True`参数，可以将分割结果展开成多个新列。`str.extract`方法则使用正则表达式从字符串中提取特定模式的内容，非常适合处理结构化的字符串数据，如从身份证号中提取出生日期。

数据变形也是常用的操作，包括长宽表转换。`melt`函数将宽表转换成长表（行数增加，列数减少），`pivot`函数则将长表转换成宽表（行数减少，列数增加）。根据分析需求选择合适的表结构，可以大大简化数据处理的复杂度。

In [18]:
# 字符串分割
data = {
    'ID': [1, 2],
    'name': ['alice smith', 'bob smith'],
    'Math': [90, 85],
    'English': [88, 92],
    'Science': [95, 89]
}
df = pd.DataFrame(data)

# 将name列分割成first和last两列
df[['first', 'last']] = df['name'].str.split(' ', expand=True)
print("分割后的数据:")
print(df)

分割后的数据:
   ID         name  Math  English  Science  first   last
0   1  alice smith    90       88       95  alice  smith
1   2    bob smith    85       92       89    bob  smith


In [19]:
# 从复合字段中提取数值
df = pd.read_csv('data/sleep.csv')
df = df[['person_id', 'blood_pressure']]
print("血压数据:")
print(df.head())

# 将血压拆分为高压和低压
df[['high', 'low']] = df['blood_pressure'].str.split('/', expand=True)
df['high'] = df['high'].astype('int64')
df['low'] = df['low'].astype('int64')

print("\n拆分后的数据:")
print(df.info())

print("\n高压平均值:", df['high'].mean())
print("低压平均值:", df['low'].mean())

血压数据:
   person_id blood_pressure
0          1         124/70
1          2         131/86
2          3         122/70
3          4         124/72
4          5         133/78

拆分后的数据:
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   person_id       400 non-null    int64
 1   blood_pressure  400 non-null    str  
 2   high            400 non-null    int64
 3   low             400 non-null    int64
dtypes: int64(3), str(1)
memory usage: 12.6 KB
None

高压平均值: 122.19
低压平均值: 73.04


In [20]:
# 数据变形：长宽表转换
data = {
    "ID": [1, 2],
    "name": ['alice', 'bob'],
    "Math": [90, 85],
    "English": [88, 92],
    "Science": [95, 89]
}
df = pd.DataFrame(data)
print("原始宽表:")
print(df)

# 转置
print("\n转置(行变列):")
print(df.T)

# 宽表转长表
df_long = pd.melt(df, id_vars=['ID', 'name'], var_name='科目', value_name='分数')
print("\n宽表转长表:")
print(df_long.sort_values('name'))

# 长表转宽表
df_wide = pd.pivot(df_long, index=['ID', 'name'], columns='科目', values='分数')
print("\n长表转宽表:")
print(df_wide)

原始宽表:
   ID   name  Math  English  Science
0   1  alice    90       88       95
1   2    bob    85       92       89

转置(行变列):
             0    1
ID           1    2
name     alice  bob
Math        90   85
English     88   92
Science     95   89

宽表转长表:
   ID   name       科目  分数
0   1  alice     Math  90
2   1  alice  English  88
4   1  alice  Science  95
1   2    bob     Math  85
3   2    bob  English  92
5   2    bob  Science  89

长表转宽表:
科目        English  Math  Science
ID name                         
1  alice       88    90       95
2  bob         92    85       89


---

## 8. 特征工程

特征工程是数据分析中最关键的步骤之一，它涉及到从原始数据中创建新的、有意义的特征，以提高模型的预测能力或更好地支持业务分析。特征工程包括但不限于：数据转换、特征构造、特征选择和特征编码等。

`rename`方法用于重命名列或索引，使数据更易理解。`set_index`和`reset_index`用于在普通列和索引之间切换。`groupby`是分组聚合的核心方法，它允许我们按照一个或多个列进行分组，然后对每个组应用聚合函数如求和、均值、计数等。

类别型变量的处理是特征工程的重要内容。将类别型变量转换为数值型有多种方法，包括标签编码（label encoding）和独热编码（one-hot encoding）。astype方法可以将字符串转换为category类型，这在机器学习任务中可以显著提高处理效率。

In [21]:
# 重命名与索引操作
df = pd.DataFrame({
    'name': ['jack', 'alice', 'tom', 'bob'],
    'age': [20, 30, 40, 50],
    'gender': ['female', 'male', 'female', 'male']
})

# 设置索引
df.set_index('name', inplace=True)
print("设置name为索引:")
print(df)

# 重置索引
df.reset_index(inplace=True)

# 重命名列
df_renamed = df.rename(columns={'age': '年龄'}, index={0: 4})
print("\n重命名后:")
print(df_renamed)

# 直接赋值修改
df.index = [1, 2, 3, 4]
df.columns = ['姓名', '年龄', '性别']
print("\n直接赋值修改后:")
print(df)

设置name为索引:
       age  gender
name              
jack    20  female
alice   30    male
tom     40  female
bob     50    male

重命名后:
    name  年龄  gender
4   jack  20  female
1  alice  30    male
2    tom  40  female
3    bob  50    male

直接赋值修改后:
      姓名  年龄      性别
1   jack  20  female
2  alice  30    male
3    tom  40  female
4    bob  50    male


In [22]:
# 分组聚合
df = pd.read_csv('data/employees.csv')
df = df.dropna(subset=['department_id'])
df['department_id'] = df['department_id'].astype('int64')

# 查看分组情况
print("分组情况:")
print(df.groupby('department_id').groups)

# 获取特定分组
print("\n部门20的员工:")
print(df.groupby('department_id').get_group(20))

# 计算各部门平均薪资
df_salary = df.groupby('department_id')[['salary']].mean()
df_salary['salary'] = df_salary['salary'].round(2)
df_salary = df_salary.reset_index()
df_salary = df_salary.sort_values('salary', ascending=False)
print("\n各部门平均薪资:")
print(df_salary)

分组情况:
{10: [100], 20: [101, 102], 30: [14, 15, 16, 17, 18, 19], 40: [103], 50: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99], 60: [3, 4, 5, 6, 7], 70: [104], 80: [45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 79], 90: [0, 1, 2], 100: [8, 9, 10, 11, 12, 13], 110: [105, 106]}

部门20的员工:
     employee_id first_name  last_name     email  phone_number  job_id  \
101          201    Michael  Hartstein  MHARTSTE  515.123.5555  MK_MAN   
102          202        Pat        Fay      PFAY  603.123.6666  MK_REP   

      salary  commission_pct  manager_id  department_id  
101  13000.0             NaN       100.0             20  
102   6000.0             NaN       201.0             20  

各部门平均薪资:
    department_id    salary
8              90  19333.33
10            110  10150.00
6  

In [23]:
# 多维度分组聚合
df_multi = df.groupby(['department_id', 'job_id'])[['salary']].mean()
df_multi = df_multi.reset_index()
df_multi['salary'] = df_multi['salary'].round(1)
df_multi = df_multi.sort_values('salary', ascending=False)
print("各部门各岗位平均薪资:")
print(df_multi)

各部门各岗位平均薪资:
    department_id      job_id   salary
13             90     AD_PRES  24000.0
14             90       AD_VP  17000.0
1              20      MK_MAN  13000.0
11             80      SA_MAN  12200.0
16            100      FI_MGR  12000.0
18            110      AC_MGR  12000.0
4              30      PU_MAN  11000.0
10             70      PR_REP  10000.0
12             80      SA_REP   8396.6
17            110  AC_ACCOUNT   8300.0
15            100  FI_ACCOUNT   7920.0
8              50      ST_MAN   7280.0
5              40      HR_REP   6500.0
2              20      MK_REP   6000.0
9              60     IT_PROG   5760.0
0              10     AD_ASST   4400.0
6              50    SH_CLERK   3215.0
7              50    ST_CLERK   2785.0
3              30    PU_CLERK   2780.0


---

## 9. 综合案例：企鹅数据分析

下面我们以企鹅数据集为例，展示完整的数据分析流程。这个案例涵盖了数据导入、缺失值处理、特征构造、分箱分析和分组聚合等各个环节，帮助读者理解如何将前面学到的知识串联起来解决实际问题。

企鹅数据集包含了不同企鹅的多种生理指标，如喙长、喙深、翼长、体重等。通过对这个数据集的分析，我们可以了解不同种类企鹅的生理特征差异，以及这些特征与性别、栖息地之间的关系。整个分析过程包括：数据探索、缺失值处理、新特征构造、分箱分组和分析总结。

In [24]:
# 综合案例：企鹅数据分析

# 1. 导入数据
import pandas as pd
import numpy as np

df = pd.read_csv('data/penguins.csv')
print("数据集前5行:")
print(df.head())

print("\n数据集基本信息:")
print(df.info())

数据集前5行:
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
3  Adelie  Torgersen             NaN            NaN                NaN   
4  Adelie  Torgersen            36.7           19.3              193.0   

   body_mass_g     sex  
0       3750.0    Male  
1       3800.0  Female  
2       3250.0  Female  
3          NaN     NaN  
4       3450.0  Female  

数据集基本信息:
<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   f

In [25]:
# 2. 数据清洗 - 缺失值处理
print("各列缺失值数量:")
print(df.isna().sum())

# 删除含缺失值的行
df.dropna(inplace=True)
print("\n删除缺失值后的数据量:", len(df))

各列缺失值数量:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

删除缺失值后的数据量: 333


In [26]:
# 3. 特征构造
# 将性别转换为类别型
df['sex'] = df['sex'].astype('category')

# 构造新特征：喙长与喙深的比值
df['bill_ratio'] = df['bill_length_mm'] / df['bill_depth_mm']

print("添加新特征后的数据:")
print(df.head())

添加新特征后的数据:
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
4  Adelie  Torgersen            36.7           19.3              193.0   
5  Adelie  Torgersen            39.3           20.6              190.0   

   body_mass_g     sex  bill_ratio  
0       3750.0    Male    2.090909  
1       3800.0  Female    2.270115  
2       3250.0  Female    2.238889  
4       3450.0  Female    1.901554  
5       3650.0    Male    1.907767  


In [27]:
# 4. 数据分析

# 体重分箱：分为低、中、高三个等级
labels = ['低', '中', '高']
df['mass_level'] = pd.cut(df['body_mass_g'], bins=3, labels=labels)
print("体重等级分布:")
print(df['mass_level'].value_counts())

# 按性别和岛屿分组分析
print("\n按性别和岛屿分组统计:")
result = df.groupby(['sex', 'island']).agg({
    'body_mass_g': ['mean', 'count']
})
print(result)

体重等级分布:
mass_level
低    150
中    128
高     55
Name: count, dtype: int64

按性别和岛屿分组统计:
                  body_mass_g      
                         mean count
sex    island                      
Female Biscoe     4319.375000    80
       Dream      3446.311475    61
       Torgersen  3395.833333    24
Male   Biscoe     5104.518072    83
       Dream      3987.096774    62
       Torgersen  4034.782609    23


---

## 10. 综合案例：睡眠质量分析

这个案例使用睡眠质量数据集，展示更复杂的数据分析流程。数据集包含了个人睡眠时长、睡眠质量、压力水平、血压等多维度的健康指标。我们将演示如何通过特征工程和分组分析，发现不同人群的睡眠模式差异。

分析过程包括：对数值特征进行分箱以创建分类变量、提取复合字段中的关键信息、计算不同BMI类别和年龄段人群的睡眠指标平均值。这些分析结果可以为改善睡眠质量提供数据支持。

In [28]:
# 综合案例：睡眠质量分析

# 1. 导入数据
import pandas as pd
import numpy as np

df = pd.read_csv('data/sleep.csv')
print("睡眠数据前5行:")
print(df.head())

print("\n数据基本信息:")
print(df.info())

print("\n数据统计描述:")
print(df.describe())

睡眠数据前5行:
   person_id  gender  age     occupation  sleep_duration  sleep_quality  \
0          1    Male   29   Manual Labor             7.4            7.0   
1          2  Female   43        Retired             4.2            4.9   
2          3    Male   44        Retired             6.1            6.0   
3          4    Male   29  Office Worker             8.3           10.0   
4          5    Male   67        Retired             9.1            9.5   

   physical_activity_level  stress_level bmi_category blood_pressure  \
0                       41             7        Obese         124/70   
1                       41             5        Obese         131/86   
2                      107             4  Underweight         122/70   
3                       20            10        Obese         124/72   
4                       19             4   Overweight         133/78   

   heart_rate  daily_steps sleep_disorder  
0          91         8539            NaN  
1          81      

In [29]:
# 2. 数据清洗
print("缺失值统计:")
print(df.isna().sum())

# 删除不需要的列
df.drop(columns='sleep_disorder', inplace=True)

缺失值统计:
person_id                    0
gender                       0
age                          0
occupation                   0
sleep_duration               0
sleep_quality                0
physical_activity_level      0
stress_level                 0
bmi_category                 0
blood_pressure               0
heart_rate                   0
daily_steps                  0
sleep_disorder             290
dtype: int64


In [30]:
# 3. 特征工程
# 类别型转换
df['gender'] = df['gender'].astype('category')
df['occupation'] = df['occupation'].astype('category')
df['bmi_category'] = df['bmi_category'].astype('category')

# 从血压字段提取高压和低压
df[['high', 'low']] = df['blood_pressure'].str.split('/', expand=True)

# 睡眠质量分箱
quality_labels = ['差', '中', '优']
df['quality_level'] = pd.cut(df['sleep_quality'], bins=3, labels=quality_labels)

# 年龄分箱
age_labels = ['青少年', '中年', '老年']
df['age_level'] = pd.cut(df['age'], bins=3, labels=age_labels)

print("特征工程后的数据:")
print(df.head())

特征工程后的数据:
   person_id  gender  age     occupation  sleep_duration  sleep_quality  \
0          1    Male   29   Manual Labor             7.4            7.0   
1          2  Female   43        Retired             4.2            4.9   
2          3    Male   44        Retired             6.1            6.0   
3          4    Male   29  Office Worker             8.3           10.0   
4          5    Male   67        Retired             9.1            9.5   

   physical_activity_level  stress_level bmi_category blood_pressure  \
0                       41             7        Obese         124/70   
1                       41             5        Obese         131/86   
2                      107             4  Underweight         122/70   
3                       20            10        Obese         124/72   
4                       19             4   Overweight         133/78   

   heart_rate  daily_steps high low quality_level age_level  
0          91         8539  124  70         

In [31]:
# 4. 数据统计分析

# BMI类别分布
print("BMI类别分布:")
print(df['bmi_category'].value_counts())

BMI类别分布:
bmi_category
Overweight     109
Underweight    102
Obese           98
Normal          91
Name: count, dtype: int64


In [32]:
# 按年龄段和BMI分组统计
result = df.groupby(['age_level', 'bmi_category']).agg({
    'sleep_duration': 'mean',
    'sleep_quality': 'mean',
    'stress_level': 'mean'
})
print("不同年龄段和BMI的睡眠指标:")
print(result)

不同年龄段和BMI的睡眠指标:
                        sleep_duration  sleep_quality  stress_level
age_level bmi_category                                             
青少年       Normal              8.100000       6.332000      4.860000
          Obese               8.250000       6.253448      5.534483
          Overweight          8.214286       6.171429      5.317460
          Underweight         7.603279       5.883607      5.426230
中年        Normal              7.422222       6.650000      4.944444
          Obese               7.805556       6.216667      5.888889
          Overweight          8.246154       5.956410      5.974359
          Underweight         8.497500       5.907500      5.750000
老年        Normal              7.420000       4.240000      4.200000
          Obese               7.900000       5.025000      8.000000
          Overweight          8.971429       6.285714      6.714286
          Underweight        10.500000       6.200000      6.000000


---

## 总结

本notebook系统介绍了Pandas数据分析的核心知识点，涵盖了数据导入导出、缺失值处理、时间数据操作、数据类型转换、重复值处理、数据分箱、字符串处理和特征工程等关键技术。通过企鹅数据分析和睡眠质量分析两个综合案例，展示了如何将这些技术应用于实际问题中。

掌握这些技能后，你将能够独立完成大部分数据分析任务，为进一步的机器学习和深度学习打下坚实基础。